# Heart Disease Prediction — Model Development & Evaluation
**CSE570 Skill Based Assignment**

Covers rubric Section 2(c) — Model Development and Evaluation (15 marks).

**Workflow (leakage-safe):**
1. Stratified 80/20 train/test split; test set is touched only at the end
2. Baseline (Dummy classifier) with full metrics
3. Six models compared by 5-fold stratified CV on the training set
4. Hyperparameter tuning with GridSearchCV (scoring = F1)
5. **Final model selected using CV score only** (never the test set)
6. Final model evaluated once on the test set, with bootstrap 95% confidence intervals
7. Interpretation, limitations, model persistence



## 1. Setup, Split, and Preprocessing Pipeline


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay)

sns.set_theme(style='whitegrid')

df = pd.read_csv('heart_disease_clean.csv')
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

X = df.drop(columns='target')
y = df['target']

# Same leakage-safe split as in phase 1
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
])

def make_pipeline(model):
    return Pipeline([('prep', preprocessor), ('model', model)])

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 2. Baseline — Dummy Classifier
The performance floor: always predicting the majority class. Accuracy alone is misleading here, so all metrics are reported. A majority-class predictor has **recall = 0** for the disease class, which is exactly why accuracy is not enough for this problem.

In [ ]:
dummy = make_pipeline(DummyClassifier(strategy='most_frequent'))
dummy.fit(X_train, y_train)
yd = dummy.predict(X_test)

baseline_df = pd.DataFrame([{
    'model': 'Baseline (majority class)',
    'accuracy':  accuracy_score(y_test, yd),
    'precision': precision_score(y_test, yd, zero_division=0),
    'recall':    recall_score(y_test, yd, zero_division=0),
    'f1':        f1_score(y_test, yd, zero_division=0),
    'roc_auc':   0.5,   # constant predictor = chance level
}]).round(3)
baseline_df

## 3. Model Zoo — All Six Classifiers
Each model uses the same preprocessing pipeline. 5-fold stratified CV on the **training set only** gives a robust comparison before the test set is touched.

**Why scaling matters:** Logistic Regression, KNN and SVM are distance/gradient-based and need standardized features; Decision Tree, Random Forest and Naive Bayes are scale-invariant, so sharing one pipeline is harmless and keeps the workflow uniform and leakage-free.

`SVC(probability=True)` is required so SVM can output probabilities for ROC-AUC and ROC curves.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'KNN':                 KNeighborsClassifier(),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(random_state=42),
    'SVM (RBF)':           SVC(probability=True, random_state=42),
    'Naive Bayes':         GaussianNB(),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'acc': 'accuracy', 'prec': 'precision', 'rec': 'recall', 'f1': 'f1', 'auc': 'roc_auc'}

cv_rows = []
for name, mdl in models.items():
    r = cross_validate(make_pipeline(mdl), X_train, y_train, cv=cv, scoring=scoring)
    cv_rows.append([name] + [r[f'test_{k}'].mean() for k in scoring] + [r['test_f1'].std()])

cv_df = pd.DataFrame(cv_rows, columns=['model', 'cv_accuracy', 'cv_precision', 'cv_recall',
                                       'cv_f1', 'cv_roc_auc', 'cv_f1_std']).round(3)
cv_df.sort_values('cv_f1', ascending=False).reset_index(drop=True)

In [ ]:
# Auto-generated reading of the untuned CV comparison
ranked = cv_df.sort_values('cv_f1', ascending=False).reset_index(drop=True)
print(f"Best untuned model by CV F1 : {ranked.loc[0,'model']} ({ranked.loc[0,'cv_f1']:.3f})")
print(f"Weakest untuned model       : {ranked.loc[len(ranked)-1,'model']} ({ranked.loc[len(ranked)-1,'cv_f1']:.3f})")
print(f"Most stable (lowest F1 std) : {cv_df.loc[cv_df.cv_f1_std.idxmin(),'model']} ({cv_df.cv_f1_std.min():.3f})")

**Your interpretation:** compare the models using the table above (e.g. a single Decision Tree usually has higher variance; Naive Bayes is limited by its independence assumption). Write this from your actual numbers.

## 4. Hyperparameter Tuning — GridSearchCV
Small grids per model (tiny dataset, so cheap). Scoring = **F1**, because missing a sick patient (false negative) is the costly error — matching the recall-first objective in the problem statement. Tuning uses the training set only.

In [ ]:
param_grids = {
    'Logistic Regression': {'model__C': [0.01, 0.1, 1, 10, 100]},
    'KNN':                 {'model__n_neighbors': [3, 5, 7, 9, 11],
                            'model__weights': ['uniform', 'distance']},
    'Decision Tree':       {'model__max_depth': [2, 3, 4, 5, None],
                            'model__min_samples_split': [2, 5, 10]},
    'Random Forest':       {'model__n_estimators': [100, 300, 500],
                            'model__max_depth': [2, 3, 4, 5, None]},
    'SVM (RBF)':           {'model__C': [0.1, 1, 10, 100],
                            'model__gamma': ['scale', 'auto', 0.01, 0.1]},
    'Naive Bayes':         {},   # no key hyperparameters to tune
}

best_estimators, best_params, tune_rows = {}, {}, []
for name, mdl in models.items():
    grid = GridSearchCV(make_pipeline(mdl), param_grids[name],
                        cv=cv, scoring='f1', n_jobs=-1)
    grid.fit(X_train, y_train)
    best_estimators[name] = grid.best_estimator_
    best_params[name] = grid.best_params_
    tune_rows.append([name, grid.best_score_, grid.best_params_])
    print(f'{name:22s} best CV F1 = {grid.best_score_:.3f}   params = {grid.best_params_}')

tuning_df = pd.DataFrame(tune_rows, columns=['model', 'tuned_cv_f1', 'best_params'])
tuning_df['tuned_cv_f1'] = tuning_df['tuned_cv_f1'].round(3)
tuning_df.sort_values('tuned_cv_f1', ascending=False).reset_index(drop=True)

## 5. Final Model Selection — by Cross-Validation Only
The final model is the one with the highest **tuned CV F1 on the training data**. The test set plays no role in this choice, which avoids selection leakage. Ties are broken in favour of the simpler/more interpretable model: if the top CV scores are within one standard error of each other, mention this in the report and justify your preference.

In [ ]:
best_name = tuning_df.sort_values('tuned_cv_f1', ascending=False).iloc[0]['model']
final_model = best_estimators[best_name]
print(f'Final model (chosen by CV F1): {best_name}')
print(f'Best hyperparameters         : {best_params[best_name]}')

## 6. Test-Set Evaluation — All Required Metrics
All tuned models are evaluated on the held-out 20% test set **for reporting and comparison only**. The final model was already fixed in Section 5.

In [ ]:
rows = []
for name, pipe in best_estimators.items():
    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    rows.append([name,
                 accuracy_score(y_test, y_pred),
                 precision_score(y_test, y_pred),
                 recall_score(y_test, y_pred),
                 f1_score(y_test, y_pred),
                 roc_auc_score(y_test, y_proba)])

test_df = pd.DataFrame(rows, columns=['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']).round(3)
test_df['selected'] = test_df['model'] == best_name
comparison = pd.concat([baseline_df.assign(selected=False), test_df], ignore_index=True)
comparison.to_csv('model_comparison_test.csv', index=False)
comparison

### 6.1 Confusion matrices


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (name, pipe) in zip(axes.ravel(), best_estimators.items()):
    ConfusionMatrixDisplay.from_estimator(pipe, X_test, y_test, ax=ax,
                                          cmap='Blues', colorbar=False)
    ax.set_title(name + (' (selected)' if name == best_name else ''))
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150)
plt.show()

### 6.2 ROC curves comparison


In [ ]:
plt.figure(figsize=(7, 6))
for name, pipe in best_estimators.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, name=name, ax=plt.gca())
plt.plot([0, 1], [0, 1], 'k--', label='Chance')
plt.title('ROC curves — test set')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150)
plt.show()

### 6.3 Detailed report for the final model


In [ ]:
y_pred_final = final_model.predict(X_test)
y_prob_final = final_model.predict_proba(X_test)[:, 1]
print(f'Final model: {best_name}')
print(classification_report(y_test, y_pred_final, target_names=['No disease', 'Disease']))
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_final).ravel()
print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp}  -> {fn} sick patient(s) missed out of {fn+tp}')

### 6.4 Bootstrap 95% confidence intervals
The test set is small (~60 rows), so a single point estimate is noisy. We resample the test set 2000 times and report percentile 95% CIs — this is the confidence-based analysis for the modeling section.

In [ ]:
rng = np.random.default_rng(42)
yt = y_test.values
n = len(yt)
boot = {'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'roc_auc': []}
for _ in range(2000):
    idx = rng.integers(0, n, n)
    if len(np.unique(yt[idx])) < 2:
        continue
    yp, ypr = y_pred_final[idx], y_prob_final[idx]
    boot['accuracy'].append(accuracy_score(yt[idx], yp))
    boot['precision'].append(precision_score(yt[idx], yp, zero_division=0))
    boot['recall'].append(recall_score(yt[idx], yp, zero_division=0))
    boot['f1'].append(f1_score(yt[idx], yp, zero_division=0))
    boot['roc_auc'].append(roc_auc_score(yt[idx], ypr))

ci_df = pd.DataFrame({
    'point_estimate': [accuracy_score(yt, y_pred_final), precision_score(yt, y_pred_final),
                       recall_score(yt, y_pred_final), f1_score(yt, y_pred_final),
                       roc_auc_score(yt, y_prob_final)],
    'ci_lower_2.5%': [np.percentile(v, 2.5) for v in boot.values()],
    'ci_upper_97.5%': [np.percentile(v, 97.5) for v in boot.values()],
}, index=boot.keys()).round(3)
ci_df.to_csv('final_model_bootstrap_ci.csv')
ci_df

## 7. Model Interpretation — Why the Model Decides
Feature importance (Random Forest) and coefficients (Logistic Regression) give clinically interpretable signals. Compare them with the EDA findings in the EDA notebook — consistency between the two is a strong point for the viva. (Both are shown regardless of which model won, since they are used for interpretation, not selection.)

In [ ]:
feat_names = best_estimators['Random Forest'].named_steps['prep'].get_feature_names_out()
rf_imp = pd.Series(best_estimators['Random Forest'].named_steps['model'].feature_importances_, index=feat_names)
rf_imp.sort_values().plot.barh(figsize=(8, 6), title='Random Forest feature importance')
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=150)
plt.show()
print('Top 5 RF features:', list(rf_imp.sort_values(ascending=False).head(5).index))

In [ ]:
lr_coef = pd.Series(best_estimators['Logistic Regression'].named_steps['model'].coef_[0], index=feat_names)
lr_coef.sort_values().plot.barh(figsize=(8, 6), title='Logistic Regression coefficients (log-odds)')
plt.tight_layout()
plt.savefig('lr_coefficients.png', dpi=150)
plt.show()
print('Top 5 LR |coef| features:', list(lr_coef.abs().sort_values(ascending=False).head(5).index))

**Your interpretation:** note which features both models agree on, and relate them to the EDA results (e.g. thal, ca, cp, oldpeak, thalach).

## 8. Save the Final Model


In [ ]:
joblib.dump(final_model, 'best_heart_disease_model.joblib')
print(f'Saved: best_heart_disease_model.joblib  ({best_name})')

## 8b. Save all tuned models
In addition to the selected final model, each of the other tuned models
is saved to disk so any one can be reloaded or compared later.


In [ ]:
import os
os.makedirs('models', exist_ok=True)
for name, pipe in best_estimators.items():
    safe = name.replace(' ', '_')
    path = f'models/{safe}.joblib'
    joblib.dump(pipe, path)
    print(f'Saved {path}')

# Helper: load any saved tuned model later (including the final model)
def load_model(name):
    safe = name.replace(' ', '_')
    return joblib.load(f'models/{safe}.joblib')


## 9. Limitations and Improvement Possibilities
- **Small sample (297 rows after cleaning):** high variance in test estimates; CV and bootstrap CIs quantify but do not remove this
- **Single cohort (Cleveland, 1988):** population drift — retraining on local hospital data is the key improvement
- **5-class clinical labels:** severity (0-4) was binarized; ordinal/multiclass models could use the full scale
- **Missing modalities:** ECG waveforms, imaging and lab history are absent — richer features would help
- **Default 0.5 threshold:** lowering the decision threshold or using class-weighted/cost-sensitive learning would increase recall where false negatives are costlier
- **Backward elimination used the full dataset** for insight only; it did not feed into any model, so evaluation is unaffected

---
